# CNN 入門ハンズオン（全8問）

**進め方**：各問題の「やること」を読み、コード中の **★の行**（その問題で自分で書く部分）を確認します。答えは先に入っているので、**「ランタイム → すべてのセルを実行」で最後まで動きます**。授業では★の行を消して、自分で書き直してみてください。

| 問題 | 内容 |
|---|---|
| 1〜2 | 画像は数字の集まり／畳み込みを手で計算する |
| 3〜8 | MNIST（手書き数字）を CNN で分類する（標準化 → 分割 → モデル → 評価 → 学習ループ → 実行） |

**メニュー →「ランタイム」→「ランタイムのタイプを変更」で CPU のままで OK です。**

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image

# 授業用の画像（sample.png）を用意します。実行するだけでOK。
_mnist = datasets.MNIST("data", train=True, download=True)
_mnist[0][0].save("sample.png")
print("準備OK")

In [ ]:
# ---- 図を描くための道具（そのまま実行）----
from matplotlib.patches import FancyBboxPatch, Rectangle

def draw_flow(labels, colors=None, figsize=(11, 2.2), loop_back=False):
    fig, ax = plt.subplots(figsize=figsize)
    n = len(labels); w = 0.8 / n * 1.0
    ax.set_xlim(0, n); ax.set_ylim(0, 1.6 if loop_back else 1); ax.axis("off")
    for i, lab in enumerate(labels):
        c = (colors or ["#dbeafe"] * n)[i]
        ax.add_patch(FancyBboxPatch((i + 0.08, 0.25), 0.84, 0.5, boxstyle="round,pad=0.02", fc=c, ec="#334155"))
        ax.text(i + 0.5, 0.5, lab, ha="center", va="center", fontsize=9)
        if i < n - 1:
            ax.annotate("", xy=(i + 1.08, 0.5), xytext=(i + 0.92, 0.5), arrowprops=dict(arrowstyle="->", lw=1.5))
    if loop_back:
        ax.annotate("", xy=(0.5, 0.8), xytext=(n - 0.5, 0.8),
                    arrowprops=dict(arrowstyle="->", lw=1.5, connectionstyle="arc3,rad=0.3", color="#b91c1c"))
        ax.text(n / 2, 1.1, "repeat for every batch", ha="center", color="#b91c1c", fontsize=9)
    plt.show()

def show_grid(ax, a, cmap="gray", vmin=None, vmax=None, fs=9, color="red"):
    ax.imshow(a, cmap=cmap, vmin=vmin, vmax=vmax)
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            ax.text(j, i, f"{a[i, j]:g}", ha="center", va="center", color=color, fontsize=fs)
    ax.set_xticks([]); ax.set_yticks([])

def show_predictions(model, loader, n=8, wrong_only=False):
    model.eval(); items = []
    with torch.no_grad():
        for x, y in loader:
            p = model(x).argmax(1)
            for k in range(len(y)):
                if (not wrong_only) or p[k] != y[k]:
                    items.append((x[k], y[k].item(), p[k].item()))
            if len(items) >= n: break
    items = items[:n]
    fig, axs = plt.subplots(1, max(len(items), 1), figsize=(1.6 * max(len(items), 1), 2))
    axs = np.atleast_1d(axs)
    for ax, (img, t, p) in zip(axs, items):
        ax.imshow(img[0] * 0.3081 + 0.1307, cmap="gray"); ax.axis("off")
        ax.set_title(f"true {t}\npred {p}", color="green" if t == p else "red", fontsize=9)
    plt.show()

↑ 図を描く道具を用意しました。実行するだけでOKです（図の文字が英語なのは、Colab で日本語が文字化けしないようにするためです）。

## 問題1：画像を読み込んで、数字の並びを見てみよう

**やること**：画像を1枚読み込み、白黒（グレースケール）に変換して、ピクセルの数字と灰色のマスの関係を確認する。

**背景**：画像は 0（真っ黒）〜255（真っ白）の数字のマス目で、その間の数字が灰色になります。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [1, 2]})
a = np.array(_mnist[0][0])
axs[0].imshow(a, cmap="gray", vmin=0, vmax=255); axs[0].set_title("28x28 image")
axs[0].add_patch(Rectangle((9.5, 9.5), 8, 8, fill=False, ec="red", lw=2)); axs[0].axis("off")
axs[1].imshow(np.linspace(0, 255, 256)[None, :], cmap="gray", aspect=20, vmin=0, vmax=255)
axs[1].set_yticks([]); axs[1].set_xticks([0, 128, 255]); axs[1].set_xticklabels(["0 (black)", "128 (gray)", "255 (white)"])
axs[1].set_title("pixel value -> brightness")
plt.show()

**図の説明**：左は 28x28 の画像で、赤枠が後で数字を見る 8x8 の部分です。右のとおり、画像の1マスは 0（黒）〜255（白）の数字で、数字が大きいほど明るくなります。

In [ ]:
# (1) 画像を開き、グレースケール（モード "L"）に変換する
img  = Image.open("sample.png")
gray = img.convert("L")          # ★ "L" = グレースケール
arr  = np.array(gray)

# (2) 形（縦, 横）と、最小値・最大値を表示する
print("形:", arr.shape)           # ★ (縦, 横)
print("最小値:", arr.min(), "最大値:", arr.max())   # ★

# (3) 左上から 8x8 マスだけ切り出す（行 10〜17, 列 10〜17）
patch = arr[10:18, 10:18]        # ★ 行10〜17, 列10〜17

# 下は表示用（そのまま実行）
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(patch, cmap="gray", vmin=0, vmax=255)
for i in range(patch.shape[0]):
    for j in range(patch.shape[1]):
        ax.text(j, i, patch[i, j], ha="center", va="center",
                color="red", fontsize=7)
plt.show()

**考えてみよう**：数字が大きいマスは明るいですか、暗いですか？ 0 のマスはどう見えますか？

## 問題2：畳み込みを手で計算しよう

**やること**：5x5 の画像に 3x3 のフィルタをかけ、特徴マップの1つ目・2つ目の値を求め、最後に全体を1つのブロックで計算する。

**背景**：「重なった数字どうしを掛けて、9個を全部足す」を、1マスずつずらして繰り返すのが畳み込みです（ストライド1）。

In [ ]:
_img = np.zeros((5, 5), int); _img[2] = 1
_flt = np.array([[-1, -1, -1], [2, 2, 2], [-1, -1, -1]])
fig, axs = plt.subplots(1, 3, figsize=(10, 3.4))
for k, (ax, c, ttl) in enumerate(zip(axs[:2], ["red", "blue"], ["position 1", "position 2"])):
    show_grid(ax, _img, cmap="Blues", vmin=0, vmax=1.6, color="black"); ax.set_title(ttl)
    ax.add_patch(Rectangle((k - 0.5, -0.5), 3, 3, fill=False, ec=c, lw=3))
show_grid(axs[2], _flt, cmap="coolwarm", color="black"); axs[2].set_title("filter 3x3")
plt.show()

**図の説明**：5x5 の画像の上に 3x3 の窓（赤＝1つ目の位置、青＝2つ目の位置）を置き、窓の中の数字と右のフィルタの数字を掛けて足します。窓を1マス右にずらしながら、これを繰り返します。

In [ ]:
image = np.array([
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])
filt = np.array([
    [-1, -1, -1],
    [ 2,  2,  2],
    [-1, -1, -1],
])

# (a) まず紙とペンで、1つ目の位置（左上 3x3）の値を計算してみよう（答えは -3）

# (b) コードで1つ目の位置を計算する
window1 = image[0:3, 0:3]
out1 = np.sum(window1 * filt)       # ★ 掛けて全部足す
print("1つ目:", out1)

# (c) 1マス右にずらした2つ目の位置を計算する
window2 = image[0:3, 1:4]           # ★ 1マス右へ
out2 = np.sum(window2 * filt)
print("2つ目:", out2)

# (d) 全体を1つのブロックで計算する（出力は 3x3 になる）
def convolve(image, filt):
    n, f = image.shape[0], filt.shape[0]
    size = n - f + 1                 # 出力サイズ（padding=0, stride=1）
    out = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            window = image[i:i+f, j:j+f]    # ★ 窓を切り出す
            out[i, j] = np.sum(window * filt)
    return out

feature_map = convolve(image, filt)
print(feature_map)

**考えてみよう**：値が大きい（6）のはどの行ですか？ このフィルタは何（横線／縦線）に反応していますか？ ReLU をかけると −3 はどうなりますか？

In [ ]:
_fm = convolve(image, filt)
fig, axs = plt.subplots(1, 3, figsize=(10, 3.4))
show_grid(axs[0], image, cmap="Blues", vmin=0, vmax=1.6, color="black"); axs[0].set_title("image 5x5")
show_grid(axs[1], _fm, cmap="coolwarm", color="black"); axs[1].set_title("feature map 3x3")
show_grid(axs[2], np.maximum(_fm, 0), cmap="coolwarm", color="black"); axs[2].set_title("after ReLU")
plt.show()

**図の説明**：5x5 の画像が 3x3 の特徴マップになりました（横線の行だけ 6 と大きい値）。ReLU をかけると −3 のような負の値が 0 になり、線に反応した場所だけが残ります。

## 問題3：データの標準化

**やること**：画素値 0〜255 を 0〜1 に直してから標準化し、MNIST データを読み込む。

**背景**：値が小さくそろっているほど CNN は計算しやすくなります（MNIST の平均は 0.1307、標準偏差は 0.3081）。

In [ ]:
_row = np.array([0, 126, 255, 126, 0]); _sc = _row / 255; _st = (_sc - 0.1307) / 0.3081
fig, axs = plt.subplots(1, 3, figsize=(10, 2.8))
for ax, v, t in zip(axs, [_row, _sc, _st], ["raw (0-255)", "/255 (0-1)", "standardized"]):
    ax.bar(range(5), v, color="#60a5fa"); ax.set_title(t); ax.axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()

**図の説明**：同じ1行の画素を、元の値（0〜255）→ 0〜1 → 標準化の順に並べました。形は変わらず、値の大きさだけが小さく、0 のまわりにそろっています。

In [ ]:
# (1) 手で確認：スライドの1行 [0, 126, 255, 126, 0] を標準化する
row = np.array([0, 126, 255, 126, 0])
scaled = row / 255                        # ★ 0〜1 にする
standardized = (scaled - 0.1307) / 0.3081  # ★ 平均を引いて標準偏差で割る
print(standardized.round(4))             # -0.4242 1.1795 2.8215 ... になればOK

# (2) 同じ処理を PyTorch で書く
transform = transforms.Compose([
    transforms.ToTensor(),                  # ★ 0〜1 に変換
    transforms.Normalize((0.1307,), (0.3081,)),   # ★ 標準化
])

train_full = datasets.MNIST("data", train=True,  download=True, transform=transform)
test_full  = datasets.MNIST("data", train=False, download=True, transform=transform)
print(len(train_full), len(test_full))     # 60000 10000

In [ ]:
_x = train_full[0][0]
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].imshow(_x[0], cmap="gray"); axs[0].set_title("after transform"); axs[0].axis("off")
axs[1].hist(_x.flatten().numpy(), bins=30, color="#60a5fa"); axs[1].set_title("pixel values (min %.2f, max %.2f)" % (_x.min(), _x.max()))
plt.tight_layout(); plt.show()

**図の説明**：変換後も画像の見た目は同じですが、画素の値は −0.42〜2.82 の範囲に変わりました。大半が背景（黒）なので、一番左の値（−0.42）に集中しています。

## 問題4：学習用とテスト用に分ける

**やること**：訓練データ15,000枚・テストデータ2,000枚だけを使い、バッチサイズを決めて DataLoader を作る。

**背景**：バッチサイズは「何枚まとめて処理してから重みを更新するか」で、今回は32枚ごとに更新します。

In [ ]:
draw_flow(['60,000 train images', 'use only 15,000', 'split into batches\nof 32 (469 batches)', 'update weights\nafter each batch'], figsize=(11, 2))

**図の説明**：60,000枚のうち15,000枚だけを使い、32枚ずつのかたまり（バッチ）に分けます。1バッチ処理するごとに重みを1回更新するので、1周（1エポック）で469回更新されます。

In [ ]:
train_set = Subset(train_full, range(15000))   # ★
test_set  = Subset(test_full,  range(2000))    # ★

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)   # ★
test_loader  = DataLoader(test_set,  batch_size=256)                 # ★

x, y = next(iter(train_loader))
print(x.shape, y.shape)   # torch.Size([32, 1, 28, 28]) torch.Size([32])

**考えてみよう**：`x.shape` の 1 は何を表していますか？（ヒント：白黒画像）

In [ ]:
x, y = next(iter(train_loader))
fig, axs = plt.subplots(1, 8, figsize=(12, 1.9))
for i, ax in enumerate(axs):
    ax.imshow(x[i, 0] * 0.3081 + 0.1307, cmap="gray"); ax.set_title(f"label {y[i].item()}"); ax.axis("off")
plt.show()

**図の説明**：train_loader から取り出した最初のバッチ（32枚）のうち8枚です。画像と正解ラベル（0〜9）が1組ずつ対応して取り出されます。

## 問題5：モデル（CNN）の設計

**やること**：畳み込み2層＋全結合1層の `SmallCNN` を完成させる。

**背景**：`__init__` で部品（構造）を決め、`forward` でデータが通る順番を決めます。

In [ ]:
draw_flow(["input\n1x28x28", "conv1+ReLU+pool\n8x14x14", "conv2+ReLU+pool\n16x7x7", "flatten\n784", "fc\n10 scores"],
          colors=["#e2e8f0", "#dbeafe", "#bfdbfe", "#fde68a", "#bbf7d0"], figsize=(11, 2.2))

**図の説明**：画像（1x28x28）が、畳み込みとプーリングを2回通るたびに小さく・枚数が多くなり（8x14x14 → 16x7x7）、最後に1列（784個）に並べて10個のスコアになります。

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)     # ★ 出力チャンネル数 8
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)    # ★ 16
        self.pool  = nn.MaxPool2d(2)                               # ★ 画像を半分に
        self.fc    = nn.Linear(16 * 7 * 7, 10)                    # ★ 10クラス（0〜9）

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # 8 x 14 x 14
        x = self.pool(torch.relu(self.conv2(x)))   # 16 x 7 x 7
        return self.fc(x.flatten(1))               # 10個のスコア

model = SmallCNN()
print(model)
print("パラメータ数:", sum(p.numel() for p in model.parameters()))   # 9,098 になればOK

**考えてみよう**：28 → 14 → 7 と画像が小さくなるのは、どの部分のおかげですか？

In [ ]:
x, _ = next(iter(train_loader))
with torch.no_grad():
    maps = torch.relu(model.conv1(x[:1]))[0]
fig, axs = plt.subplots(1, 9, figsize=(13, 1.8))
axs[0].imshow(x[0, 0], cmap="gray"); axs[0].set_title("input"); axs[0].axis("off")
for i in range(8):
    axs[i + 1].imshow(maps[i], cmap="gray"); axs[i + 1].set_title(f"map {i}"); axs[i + 1].axis("off")
plt.show()

**図の説明**：1枚の画像を conv1 に通すと、8種類のフィルタから8枚の特徴マップができます。まだ学習前でフィルタがランダムなので意味のある形ではありませんが、学習するとエッジなどに反応するようになります。

## 問題6：評価の関数を作る

**やること**：画像の予測と正解ラベルを比べて、正解率と混同行列（行＝正解、列＝予測）を返す `evaluate` を完成させる。

**背景**：モデルは各数字のスコアを出すので、いちばん大きいスコアの数字を予測とします。

In [ ]:
_p = [7, 2, 1, 0, 4]; _t = [7, 2, 1, 0, 9]
fig, ax = plt.subplots(figsize=(6, 1.8)); ax.axis("off"); ax.set_xlim(0, 6); ax.set_ylim(0, 2)
ax.text(0.3, 1.5, "pred", ha="right"); ax.text(0.3, 0.5, "true", ha="right")
for i in range(5):
    ok = _p[i] == _t[i]
    ax.add_patch(Rectangle((0.6 + i, 0.2), 0.8, 1.6, fc="#bbf7d0" if ok else "#fecaca", ec="k"))
    ax.text(1.0 + i, 1.5, _p[i], ha="center", va="center", fontsize=14); ax.text(1.0 + i, 0.5, _t[i], ha="center", va="center", fontsize=14)
plt.show()

**図の説明**：モデルの予測（上）と正解（下）を1枚ずつ比べ、一致すれば緑、違えば赤です。5枚中4枚が一致なら正解率は 0.8 で、これを全テスト画像でやるのが evaluate です。

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    confusion = torch.zeros(10, 10, dtype=torch.int64)
    with torch.no_grad():                       # 学習しないので勾配は不要
        for x, y in loader:
            pred = model(x).argmax(dim=1)       # ★ スコア最大の数字を選ぶ
            correct += (pred == y).sum().item()  # ★ 予測と正解が一致した数
            total += y.size(0)
            for t, p in zip(y, pred):
                confusion[t, p] += 1            # ★ 行＝正解、列＝予測
    return correct / total, confusion

acc, _ = evaluate(model, test_loader)
print(f"学習前の正解率: {acc:.3f}")   # 約 0.10（でたらめと同じ）

**考えてみよう**：学習前の正解率が約 0.10 なのはなぜですか？（ヒント：10クラスの当てずっぽう）

In [ ]:
show_predictions(model, test_loader, n=8)

**図の説明**：学習前のモデルでテスト画像を予測した結果です。まだ学習していないので、予測（pred）が正解（true）とほとんど合いません。

## 問題7：学習ループを作る

**やること**：「順伝播 → 損失 → 逆伝播 → 更新」の4ステップを書いて、1エポック分の学習関数を完成させる。

**背景**：予測のズレ（損失）を測り、そのズレが小さくなる向きに重みを少しずつ直すのを繰り返して学習します。

In [ ]:
draw_flow(["forward\nmodel(x)", "loss\nloss_fn(out, y)", "backward\nloss.backward()", "step\noptimizer.step()"],
          colors=["#dbeafe", "#fde68a", "#fecaca", "#bbf7d0"], figsize=(11, 2.6), loop_back=True)

**図の説明**：1バッチごとに「予測する → ズレ（損失）を測る → どちらに直せばよいか計算する → 重みを少し直す」の4ステップを行い、これを全バッチでくり返します。

In [ ]:
loss_fn   = nn.CrossEntropyLoss()                     # ★ 分類用の損失
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)   # ★

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for i, (x, y) in enumerate(loader, 1):
        optimizer.zero_grad()          # 前回の勾配を消す
        out  = model(x)                # ★ 順伝播
        loss = loss_fn(out, y)         # ★ 損失
        loss.backward()                # ★ 逆伝播
        optimizer.step()               # ★ 更新
        total_loss += loss.item()
        if i % 100 == 0:
            print(f"  バッチ {i}/{len(loader)}  loss={loss.item():.3f}")
    return total_loss / len(loader)

**考えてみよう**：`optimizer.zero_grad()` を忘れると何が起きるでしょう？

In [ ]:
w = np.linspace(-3, 3, 100); _w = 2.5; _path = [_w]
for _ in range(5):
    _w -= 0.3 * 2 * _w; _path.append(_w)
plt.figure(figsize=(5, 3)); plt.plot(w, w ** 2, "k")
plt.plot(_path, np.array(_path) ** 2, "ro-"); plt.xlabel("weight"); plt.ylabel("loss"); plt.show()

**図の説明**：重みを少しずつ動かすと、損失（ズレ）が小さくなる方向へ赤い点が進み、谷の底（損失が最小）に近づきます。optimizer.step() はこの「1歩」にあたります。

## 問題8：学習を実行して、結果を読む

**やること**：1エポック学習して、学習前後の正解率を比べ、混同行列から「どの数字を間違えやすいか」を読み取る。

**背景**：正解率が約0.93まで上がれば成功です（実行は1〜2分ほどかかります）。

In [ ]:
draw_flow(["evaluate\n(before)", "train_one_epoch", "evaluate\n(after)", "confusion matrix\n& save model"],
          colors=["#e2e8f0", "#fde68a", "#e2e8f0", "#bbf7d0"], figsize=(11, 2))

**図の説明**：学習前の正解率を測り、1エポック学習し、もう一度測って、混同行列で結果を読む、という順番で進めます。

In [ ]:
acc_before, _ = evaluate(model, test_loader)
print(f"学習前の正解率: {acc_before:.3f}")

start = time.time()
avg_loss = train_one_epoch(model, train_loader)   # ★
print(f"平均loss={avg_loss:.3f}  時間={time.time()-start:.0f}秒")

acc_after, confusion = evaluate(model, test_loader)   # ★
print(f"学習後の正解率: {acc_after:.3f}")

# 混同行列を図にする（表示用）
plt.figure(figsize=(5, 5))
plt.imshow(confusion, cmap="Blues")
plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.show()

# 最も間違えやすい組み合わせ（正解≠予測のうち最大）を探す
wrong = confusion.clone()
wrong.fill_diagonal_(0)
idx = wrong.argmax().item()
true_digit, pred_digit = idx // 10, idx % 10      # ★
print(f"最も多い間違い：正解 {true_digit} を {pred_digit} と予測（{wrong.max().item()}枚）")

torch.save(model.state_dict(), "mnist_cnn.pt")
print("OK" if acc_after >= 0.90 else "0.90未満：設定を見直そう")

**考えてみよう**：混同行列の対角線（左上〜右下）が濃いのはなぜですか？ 間違えやすい数字は、見た目がどう似ていますか？

---
### まとめ
- 画像は数字の集まり。**畳み込み**で特徴を取り出し、**プーリング**で縮める。
- 学習は **順伝播 → 損失 → 逆伝播 → 更新** のくり返し。

In [ ]:
show_predictions(model, test_loader, n=8, wrong_only=True)

**図の説明**：学習後のモデルが間違えた画像です（赤＝間違い）。人が見てもまぎらわしい字が多く、混同行列で読み取った「間違えやすい組み合わせ」と対応しています。